In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
from matplotlib.colors import Normalize

from helpers import compute_ie_influence_gain, compute_ie_params_scan

# Set style

In [ ]:
STYLE = {
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "lines.linewidth": 1,
    "axes.linewidth": 0.6,  # Axis border
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "figure.figsize": (6.89, 1.9),  # ~wide (full 17.5cm wide)
    "figure.dpi": 300
}

mpl.rcParams.update(STYLE)

# gains used throughout the figure
RHO_LOW = 0.6
RHO_HIGH = 1.2

C_LOW = '#6BA4DA'
C_HIGH = '#2C6FB0'

SIGMA_E = 10   # spread of the excitatory connections (in units of neurons)

# Initialize figure

In [ ]:
layout = [
    ['f', 'g', 'h_low', 'h_high', 'h_cbar'],
]
width_ratios = [1, 1, 1.15, 1.15, 0.07]

fig = plt.figure()

axes = fig.subplot_mosaic(
    layout,
    width_ratios=width_ratios,
    sharex=False,
    sharey=False
)

fig.subplots_adjust(left=0.08, right=0.95, top=0.82, bottom=0.25, wspace=0.75)

## Panel labels

In [ ]:
panel_labels = {
    "f": "f",
    "g": "g",
    "h_low": "h",
    "h_high": None,
    "h_cbar": None,
}

for key, ax in axes.items():
    label = panel_labels.get(key)
    if label is not None:
        ax.annotate(
            label,
            xy=(-0.45, 1.3),  # adjust x (left) and y (slightly above top)
            xycoords='axes fraction',
            fontsize=10,
            va='top',
            ha='right'
        )

plt.tight_layout()
fig

# f: LELI regime

Net I-E influence when a single inhibitory unit is perturbed, using the same
parameters as in Figure 4b (`eg_mh`).

In [ ]:
dist_mh, infl_mh = compute_ie_influence_gain('eg_mh', gains=(RHO_HIGH, RHO_LOW))

In [ ]:
name = 'f'
ax = axes[name]

for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_position(('outward', 5))
ax.spines['left'].set_position(('outward', 5))

ax.axhline(y=0.00, color='black', linestyle='--', linewidth=0.6)
ax.plot(dist_mh, infl_mh[RHO_HIGH], color=C_HIGH, linewidth=1.5,
        label=r'$\rho_{high}=%.1f$' % RHO_HIGH)
ax.plot(dist_mh, infl_mh[RHO_LOW], color=C_LOW, linewidth=1.5,
        label=r'$\rho_{low}=%.1f$' % RHO_LOW)

ax.set_xlim([0, 2.4])
ax.set_xticks([0, 2.4])
ax.set_xticklabels(['0', '2.4'])
ax.set_ylim([-0.7, 0.02])
ax.set_yticks([-0.7, 0.0])
ax.set_yticklabels(['-0.7', '0.0'])

ax.set_xlabel(r'Distance ($\sigma_E$)', labelpad=2)
ax.set_ylabel('Influence (a.u.)', labelpad=-2)
ax.set_title('LELI', pad=13)
ax.tick_params(axis='x', pad=1)
ax.tick_params(axis='y', pad=1)

ax.legend(loc='lower right', bbox_to_anchor=(1.05, -0.03), fontsize=6,
          frameon=False, handlelength=1.2, borderpad=0, labelspacing=0.3)

fig

# g: Cross-dominant suppressive regime

Same perturbation, using the same parameters as in Figure 4c (`eg_cross_pop`).

In [ ]:
dist_cd, infl_cd = compute_ie_influence_gain('eg_cross_pop', gains=(RHO_HIGH, RHO_LOW))

In [ ]:
name = 'g'
ax = axes[name]

for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_position(('outward', 5))
ax.spines['left'].set_position(('outward', 5))

ax.axhline(y=0.00, color='black', linestyle='--', linewidth=0.6)
ax.plot(dist_cd, infl_cd[RHO_HIGH], color=C_HIGH, linewidth=1.5)
ax.plot(dist_cd, infl_cd[RHO_LOW], color=C_LOW, linewidth=1.5)

ax.set_xlim([0, 2.4])
ax.set_xticks([0, 2.4])
ax.set_xticklabels(['0', '2.4'])
ax.set_ylim([-0.021, 0.01])
ax.set_yticks([-0.02, 0.0, 0.01])

ax.set_xlabel(r'Distance ($\sigma_E$)', labelpad=2)
ax.set_ylabel('Influence (a.u.)', labelpad=-2)
ax.set_title('Cross-Dominant\nSuppressive Regime', pad=4)
ax.tick_params(axis='x', pad=1)
ax.tick_params(axis='y', pad=1)

fig

# h: Net I-E influence across connection strengths

Same fixed parameters as in **g**, at low gain (*left*) and high gain (*right*).
The black dashed line delimits the region consistent with the experimental data
shown in Figure 4e.

The scan takes a few minutes.

In [ ]:
# simulate
Wee, Wei, scan = compute_ie_params_scan(gains=(RHO_LOW, RHO_HIGH))

In [ ]:
X, Y = np.meshgrid(Wee, Wei)
vmin, vmax = -0.02, 0.0

for name, gain, title in [('h_low', RHO_LOW, r'$\rho_{low}=%.1f$' % RHO_LOW),
                          ('h_high', RHO_HIGH, r'$\rho_{high}=%.1f$' % RHO_HIGH)]:
    ax = axes[name]
    avg_infl = scan[gain]['avg_ie_infl']
    stability = scan[gain]['stability']
    plausible = scan[gain]['plausible']

    c = ax.pcolormesh(X, Y, avg_infl, shading='auto', cmap='RdPu_r',
                      vmin=vmin, vmax=vmax)

    # instability region
    ax.contour(X, Y, stability, levels=[1], colors='black', linewidths=1)
    ax.text(2.0, 0.25, 'Instability', fontsize=6, color='black',
            ha='center', va='center')

    # region consistent with the experimental data (Figure 4e)
    ax.contour(X, Y, plausible.astype(float), levels=[0.5], colors='black',
               linewidths=0.6, linestyles='--')

    ax.set_xlim([0, 2.5])
    ax.set_ylim([0, 3.1])
    ax.set_xticks([0, 2.5])
    ax.set_yticks([0, 2.5])
    ax.set_xlabel('$w_{EE}$', labelpad=-2)
    ax.set_title(title, pad=4)
    ax.tick_params(axis='x', pad=1)
    ax.tick_params(axis='y', pad=1)

axes['h_low'].set_ylabel(r'$\sqrt{w_{EI}w_{IE}}$', labelpad=-2)

cbar = fig.colorbar(c, cax=axes['h_cbar'], ticks=[vmin, -0.01, vmax])
cbar.set_label('Average I-E influence (a.u.)', rotation=270, labelpad=8)
cbar.ax.tick_params(pad=1)

fig

# Save

In [ ]:
# import datetime
# fig_path = ''
# draft_id = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# fig.savefig(fig_path + "fig5_model{}.svg".format(draft_id), dpi=300, bbox_inches='tight')
# fig.savefig(fig_path + "fig5_model{}.pdf".format(draft_id), dpi=300, bbox_inches='tight')